# Web Scraping

Web data can usually be collected through one of three approaches:

- **HTML parsing:** retrieve an HTML response and extract data from its elements.
- **API requests:** call a documented or internal HTTP endpoint and parse its structured response, commonly JSON.
- **Browser automation:** run a browser when the page requires JavaScript execution, browser state, or user interaction.

Use the least complex approach that exposes the required data. A direct API is generally more stable than presentation-oriented HTML, while an HTTP request is substantially cheaper than browser automation.

## 1. HTTP requests and APIs

The [Requests](https://requests.readthedocs.io/) library sends HTTP requests without launching a browser. It returns the server's response but does not execute JavaScript.

A response contains three main components:

- A status code, such as `200 OK` or `404 Not Found`.
- Headers that describe the response and its representation.
- A body containing HTML, JSON, text, or binary data.

In [ ]:
import os
import re
import time
from io import StringIO
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

### 1.1. Sending HTTP requests

Use a timeout so that an unresponsive server does not block the program indefinitely. After receiving a response, call `raise_for_status()` to raise an exception for HTTP error responses.

In [ ]:
url = "https://books.toscrape.com/index.html"

response = requests.get(url, timeout=30)
response.raise_for_status()

{
    "url": response.url,
    "status_code": response.status_code,
    "content_type": response.headers.get("Content-Type"),
    "body_length": len(response.content),
}

In [ ]:
response.text[:1000]

In [ ]:
response.headers

Use `response.text` for decoded text such as HTML and `response.content` for raw bytes. For JSON responses, `response.json()` deserializes the body into Python objects.

Request data should be passed through the argument that matches its HTTP representation:

| Component | Requests argument | Typical use |
|---|---|---|
| URL query string | `params` | Filters, search terms, page numbers |
| Request headers | `headers` | Authentication, content negotiation, client metadata |
| Form body | `data` | HTML form submissions |
| JSON body | `json` | Structured API input |
| Cookies | `cookies` or `Session` | Session state |

In [ ]:
api_url = "https://api.github.com/repos/dmlc/xgboost/commits"
params = {
    "per_page": 5,
    "page": 1,
}
headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

response = requests.get(
    api_url,
    params=params,
    headers=headers,
    timeout=30,
)
response.raise_for_status()

commits = response.json()
pd.DataFrame(
    {
        "sha": item["sha"],
        "author": item["commit"]["author"]["name"],
        "message": item["commit"]["message"].splitlines()[0],
    }
    for item in commits
)

A `requests.Session` preserves cookies and default headers across requests and reuses network connections. It is useful when several requests target the same site.

In [ ]:
with requests.Session() as session:
    session.headers.update({"User-Agent": "data-collection-example/1.0"})

    response = session.get(url, timeout=30)
    response.raise_for_status()

    html = response.text

### 1.2. Documented APIs

Documented APIs define their endpoints, HTTP methods, parameters, authentication requirements, response schemas, pagination rules, and usage limits. Follow the documentation rather than inferring these details from examples.

The endpoint below returns the number of bytes attributed to each programming language detected in a public GitHub repository.

In [ ]:
owner = "dmlc"
repository = "xgboost"

url = f"https://api.github.com/repos/{owner}/{repository}/languages"
headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

response.json()

For a private repository, add an access token with the required repository permission. The example is guarded so that the cell does not send a request when `GITHUB_TOKEN` is unavailable.

In [ ]:
github_token = os.getenv("GITHUB_TOKEN")

if github_token:
    private_url = "https://api.github.com/repos/hungpq7/courses/languages"
    private_headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {github_token}",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    private_response = requests.get(
        private_url,
        headers=private_headers,
        timeout=30,
    )
    private_response.raise_for_status()
    private_languages = private_response.json()
else:
    private_languages = None

private_languages

The same public API request can be sent from the command line with `curl`:

In [ ]:
%%bash
curl --fail --silent --show-error \
    "https://api.github.com/repos/dmlc/xgboost/languages" \
    -H "Accept: application/vnd.github+json" \
    -H "X-GitHub-Api-Version: 2022-11-28"

APIs commonly use `GET` to retrieve a resource, `POST` to submit data or execute an operation, `PUT` or `PATCH` to update a resource, and `DELETE` to remove one. The API contract, rather than the method name alone, determines the operation.

Private endpoints require credentials with appropriate permissions. Never store tokens in a notebook or commit them to a repository. Load secrets from environment variables or a secrets manager:

```python
import os

headers = {
    "Authorization": f"Bearer {os.environ['API_TOKEN']}",
}
```

### 1.3. Finding internal APIs

Many client-rendered sites retrieve data after the initial HTML response. The browser may obtain this data from internal HTTP endpoints and use it to update the page. These endpoints are not necessarily public or stable, but the browser's network log reveals the requests required by the page.

A practical inspection workflow is:

1. Open the target page and the browser's developer tools, then select the **Network** panel.
2. Reload the page so that the panel captures requests from the beginning.
3. Perform the action that loads the required data, such as searching, changing a filter, or moving to the next page.
4. Filter for **Fetch/XHR** requests and inspect likely entries. Fetch/XHR responses may contain JSON, HTML, text, or another format.
5. In **Headers**, record the request URL, HTTP method, query parameters, required headers, cookies, and request body.
6. In **Preview** or **Response**, verify that the response contains the required records.
7. Reproduce the smallest necessary request in Python and compare its response with the browser response.

**Copy as cURL** is useful for reproducing a request initially. Remove browser-specific and transient headers one at a time to determine which values are actually required.

The structure of a reproduced request usually looks like this:

```python
url = "https://example.com/api/items"
params = {
    "page": 1,
    "limit": 100,
}
headers = {
    "Accept": "application/json",
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=30,
)
response.raise_for_status()
items = response.json()
```

Some endpoints require cookies, anti-forgery tokens, signed parameters, or browser-generated state. In those cases, use a `Session` or browser automation rather than assuming that a `User-Agent` header is sufficient.

Internal endpoints can change without notice. Check the site's terms, access rules, and rate limits before relying on them.

#### Case: TechCrunch

The original browser inspection identified a TechCrunch endpoint named `magazine`. After confirming its response in the Network panel, the request can be reproduced directly.

:::{image} ../image/chap_01/rest_api_response.png
:height: 300px
:align: center
:::

:::{image} ../image/chap_01/rest_api_url.png
:height: 300px
:align: center
:::

Internal endpoints and response fields may change. If this example stops working, repeat the inspection workflow instead of assuming that the historical URL remains valid.

In [ ]:
url = (
    "https://techcrunch.com/wp-json/tc/v1/magazine"
    "?page=1&_embed=true&cachePrevention=0"
)

response = requests.get(url, timeout=30)
response.raise_for_status()
articles = response.json()

In [ ]:
data = []
for item in articles:
    primary_category = item.get("primary_category") or {}
    parsely_metadata = item.get("parselyMeta") or {}
    authors = parsely_metadata.get("parsely-author") or []

    data.append(
        {
            "id": item.get("id"),
            "category": primary_category.get("slug"),
            "author": authors[0] if authors else None,
            "title": parsely_metadata.get("parsely-title"),
        }
    )

pd.DataFrame(data).head()

#### Case: Tiki

The original inspection of Tiki's book category found a `listings` endpoint. The browser request included a `User-Agent` header and URL query parameters.

:::{image} ../image/chap_01/rest_api_payload.png
:height: 300px
:align: center
:::

A `User-Agent` may be sufficient for a particular endpoint, but it is not a general way to bypass access controls. Other endpoints may require cookies, tokens, signatures, or browser-generated state.

In [ ]:
requests.utils.default_headers()

In [ ]:
url = (
    "https://tiki.vn/api/personalish/v1/blocks/listings"
    "?limit=40&category=8322&page=1&urlKey=nha-sach-tiki"
)
headers = {
    "User-Agent": "Mozilla/5.0 Chrome/108.0.0.0 Safari/537.36",
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

The URL is easier to read and modify when its query parameters are passed separately through `params`:

In [ ]:
url = "https://tiki.vn/api/personalish/v1/blocks/listings"
headers = {
    "User-Agent": "Mozilla/5.0 Chrome/108.0.0.0 Safari/537.36",
}
params = {
    "limit": 40,
    "category": 8322,
    "page": 1,
    "urlKey": "nha-sach-tiki",
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30,
)
response.raise_for_status()

In [ ]:
for product in response.json()["data"][:5]:
    print(product["name"])

### 1.4. Pagination and error handling

APIs commonly paginate through a page number, an offset, a cursor, or a URL supplied in the response. Stop according to the documented termination condition, such as an empty result, a missing next-page link, or a null cursor. Do not interpret every non-`200` response as the end of pagination; authentication failures, rate limits, and server errors require separate handling.

In [ ]:
def iter_github_commits(
    owner: str,
    repository: str,
    per_page: int = 100,
):
    """Yield commit objects from every available result page."""
    url = f"https://api.github.com/repos/{owner}/{repository}/commits"
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    params = {"per_page": per_page}

    with requests.Session() as session:
        while url is not None:
            response = session.get(
                url,
                params=params,
                headers=headers,
                timeout=30,
            )
            response.raise_for_status()

            yield from response.json()

            url = response.links.get("next", {}).get("url")
            params = None  # The next URL already contains its query string.

For production collection code, also consider retries with exponential backoff for transient failures, explicit handling for `429 Too Many Requests`, server-provided rate-limit headers, logging, and checkpoints that allow an interrupted job to resume.

## 2. HTML parsing

HTML parsing is appropriate when the required data is present in the HTTP response. If the data appears only after JavaScript runs, first look for the underlying API; otherwise, use browser automation.

### 2.1. HTML structure

An HTML document contains nested elements. Most elements have an opening tag, content, and a closing tag; void elements such as `<br>` have no closing tag.

```html
<article class="product" data-product-id="42">
  <h2 id="title">Example book</h2>
  <a href="/books/42">Details</a>
</article>
```

In this fragment:

- `article`, `h2`, and `a` are tags.
- `Example book` and `Details` are text content.
- `class`, `data-product-id`, `id`, and `href` are attributes.
- The elements form a tree in which `h2` and `a` are children of `article`.

IDs should be unique within a document. Classes identify groups of elements and may contain multiple tokens. Data attributes store element-specific values.

In [ ]:
%%html
<span class="breadcrumb content" style="color: indianred;">
  computer
</span>

### 2.2. Parsing and selecting elements

[Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) parses HTML into a tree that can be searched and traversed. The built-in `html.parser` requires no additional parser dependency; `lxml` is another common choice when installed.

In [ ]:
html = """
<html>
  <head>
    <title>The Dormouse's story</title>
  </head>
  <body>
    <p class="story">
      Once upon a time there were three little sisters:
      <a href="https://example.com/elsie" class="sister" id="link1">Elsie</a>,
      <a href="http://example.com/lacie" class="sister" id="link2">Lacie</a>,
      and
      <a href="http://example.com/tillie" class="sister" id="link3">Tillie</a>.
    </p>
    <time class="story">2000-01-01 06:00:00</time>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")

#### Tree navigation

Attribute-style navigation follows the first matching child at each level. It is concise when the structure is known and only one match is required.

In [ ]:
first_link = soup.body.p.a
first_link

In [ ]:
first_link.get_text(strip=True)

In [ ]:
first_link["href"]

Beautiful Soup supports both method-based searches and CSS selectors:

| Method | Result |
|---|---|
| `find(...)` | First matching element or `None` |
| `find_all(...)` | List of all matching elements |
| `select_one("...")` | First CSS-selector match or `None` |
| `select("...")` | List of all CSS-selector matches |

Use `class_` for the HTML `class` attribute because `class` is a Python keyword. Use the `attrs` dictionary for attributes whose names are not valid Python identifiers:

```python
soup.find("article", attrs={"data-product-id": "42"})
```

CSS selectors express compound conditions compactly:

```css
article.product h2
article[data-product-id="42"]
nav.pagination li.next > a
```

#### Element searching

Tag names, IDs, classes, regular expressions, and attribute dictionaries can be combined to express different searches.

In [ ]:
soup.find(id="link2")

In [ ]:
soup.find_all(re.compile(r"^t"))

In [ ]:
soup.find_all("a", class_="sister")

In [ ]:
attrs = {
    "class": "sister",
    "href": re.compile(r"https\S+"),
}

soup.find_all("a", attrs=attrs)

Extract values deliberately:

- `element.get_text(" ", strip=True)` returns normalized text.
- `element["href"]` requires the attribute and raises `KeyError` if it is absent.
- `element.get("href")` returns `None` when the attribute is absent.
- `urljoin(base_url, relative_url)` converts a relative link to an absolute URL.

The following example extracts books from one category page.

In [ ]:
page_url = (
    "https://books.toscrape.com/"
    "catalogue/category/books/fiction_10/index.html"
)

response = requests.get(page_url, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

books = []
for card in soup.select("article.product_pod"):
    link = card.select_one("h3 > a")
    rating_classes = card.select_one("p.star-rating").get("class", [])
    rating = next(
        (value for value in rating_classes if value != "star-rating"),
        None,
    )

    books.append(
        {
            "title": link["title"],
            "price": card.select_one("p.price_color").get_text(strip=True),
            "rating": rating,
            "url": urljoin(page_url, link["href"]),
        }
    )

pd.DataFrame(books).head()

Prefer selectors based on stable semantics or document structure. Generated class names and absolute XPath expressions are usually brittle. When an element is optional, check for `None` before accessing its text or attributes.

### 2.3. Pagination and detail pages

Multi-page HTML scraping usually has two phases:

1. Traverse listing pages and collect canonical item URLs.
2. Request each item page and extract its detailed fields.

Follow the page's next link when possible instead of constructing page numbers or treating every HTTP error as the final page.

The category also exposes a numeric URL pattern. This alternative increments `page-{n}` and stops only on a `404 Not Found` response. Other HTTP errors are raised rather than silently interpreted as the end of pagination.

In [ ]:
def collect_book_urls_by_page_number(
    category_url: str,
) -> list[str]:
    """Collect book URLs by incrementing the page number."""
    book_urls = []
    page_number = 1

    with requests.Session() as session:
        while True:
            page_url = category_url.replace(
                "index",
                f"page-{page_number}",
            )
            response = session.get(page_url, timeout=30)

            if response.status_code == 404:
                break

            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            containers = soup.select(
                "li.col-xs-6.col-sm-4.col-md-3.col-lg-3"
            )
            book_urls.extend(
                urljoin(page_url, container.select_one("h3 > a")["href"])
                for container in containers
            )

            page_number += 1

    return book_urls

In [ ]:
category_url = (
    "https://books.toscrape.com/"
    "catalogue/category/books/fiction_10/index.html"
)

numbered_book_urls = collect_book_urls_by_page_number(category_url)
len(numbered_book_urls)

In [ ]:
def collect_book_urls(
    category_url: str,
    delay: float = 0.5,
) -> list[str]:
    """Collect all book URLs from a category and its pagination."""
    book_urls = []
    page_url = category_url

    with requests.Session() as session:
        while page_url is not None:
            response = session.get(page_url, timeout=30)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            book_urls.extend(
                urljoin(page_url, link["href"])
                for link in soup.select("article.product_pod h3 > a")
            )

            next_link = soup.select_one("li.next > a")
            page_url = (
                urljoin(page_url, next_link["href"])
                if next_link is not None
                else None
            )

            if page_url is not None:
                time.sleep(delay)

    return book_urls

In [ ]:
book_urls = collect_book_urls(category_url)
len(book_urls)

In [ ]:
def parse_currency(value: str) -> float:
    """Extract the numeric component of a price string."""
    match = re.search(r"\d+\.\d+", value)
    if match is None:
        raise ValueError(f"Could not parse currency value: {value!r}")
    return float(match.group())


def scrape_book_details(book_url: str) -> dict:
    """Extract selected fields from one Books to Scrape detail page."""
    response = requests.get(book_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    product = soup.select_one("article.product_page")
    product_main = product.select_one("div.product_main") if product else None
    if product is None or product_main is None:
        raise ValueError(f"Product container not found: {book_url}")

    description_heading = product.select_one("#product_description")
    description = (
        description_heading.find_next_sibling("p").get_text(" ", strip=True)
        if description_heading is not None
        else None
    )

    table_element = product.select_one("table.table-striped")
    if table_element is None:
        raise ValueError(f"Product table not found: {book_url}")

    table = pd.read_html(StringIO(str(table_element)))[0]
    properties = table.set_index(0)[1]

    rating_classes = product_main.select_one("p.star-rating").get(
        "class",
        [],
    )
    rating = next(
        (value for value in rating_classes if value != "star-rating"),
        None,
    )

    return {
        "upc": properties.get("UPC"),
        "title": product_main.select_one("h1").get_text(" ", strip=True),
        "description": description,
        "price_excl_tax": parse_currency(properties["Price (excl. tax)"]),
        "tax": parse_currency(properties["Tax"]),
        "availability": product_main.select_one(
            "p.availability"
        ).get_text(" ", strip=True),
        "rating": rating,
        "url": book_url,
    }

In [ ]:
# Limit the example to five detail pages.
records = [scrape_book_details(url) for url in book_urls[:5]]
book_df = pd.DataFrame(records)

book_df

Keep collection logic separate from data transformation and storage. For larger jobs, add a delay between requests, reuse a `Session`, log failed URLs, checkpoint results, and validate the extracted schema. Respect `robots.txt`, terms of service, authentication boundaries, and published rate limits.